In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile
import ipywidgets as W
from ipywidgets import interact, widgets, fixed

# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from scene import generate_single_cell_3d
from render import N_POOLS
from tape import Tape
from plot import plot_surface_xyz_inline, plot_surface_xyz_html

%matplotlib inline

# 1) Synthetic cell generation

In [ ]:
SEED=None

K=20
SIZE = 201
TILE = 256
N_CAND = 1500

# Draw 3DTape. For the later optimization, we want to draw the random numbers once, and then use them for all parameter combinations. This is to avoid the random numbers changing when we change parameters.
# instead of a 2d size, now its a 3d volume
VOL=(128, 128, 128)
# Ca 10 pixels per ym VERIFY!
UM_PER_VOX = 10
# How much do we want to downsample the z scale?
SPACING = (4, 1, 1)
# Degrees of freedom for the sperical harmonics. l is the angular frequency (i.e. how many lobes around the 3d cell)
L_MIN = 2
L = 4
RADIUS = 24.0
ROUGH = 0.25
BETA = 1.9
NUC_FRAC = 0.25
ELONG = 1.6
POLAR_DEG = 65
AZIM_DEG = 25
ROLL_DEG = 0
RIM = 1.5
NUC_CORR = 0.4
NUC_OFFSET = 0.6


tape = Tape(seed=SEED, size=SIZE, K=K)
tape.draw(tile = TILE, n_cand = N_CAND, Pool = N_POOLS)
tape.draw3d(vol = VOL, n_cand = N_CAND, Pool = N_POOLS, l_min = L_MIN, L = L)

In [ ]:
from dataclasses import dataclass

# Verify
FLUOROPHORES = {
    "DAPI": 0.461, "FITC": 0.519, "PE": 0.578, "APC": 0.660
}

# Optics: 
@dataclass(frozen=True)
class Optics:
    """Known Physical properties of the detector (Macsima) and the experiment."""
    um_per_px: float = 0.325 
    focal_um: float = 0.0
    
    # VERIFY
    na: float = 0.75 #or 0.45 
    wavelength_um: float = 0.530 
    
    n_immersion: float = 1.0
    
    @property
    def tan_theta(self):
        return float(np.tan(np.arcsin(np.clip(self.na / self.n_immersion, 0.0, 0.999))))

## 1.1) Cell shape gen

In [ ]:
cell = generate_single_cell_3d(tape, i=0, size=VOL, radius=RADIUS, nuc_frac=NUC_FRAC, rough=ROUGH, elong=ELONG, beta=BETA,
          polar_deg=PÖLAR_DEG, azim_deg=AXIM_DEG, roll_deg=ROLL_DEG, rim=RIM, nuc_corr=NUC_CORR, nuc_offset=NUC_OFFSET)

In [ ]:
fig = plot_surface_xyz_inline(cell = cell, elong = ELONG, 
                        polar_deg = POLAR_DEG, azim_deg = AZIM_DEG, roll_deg = ROLL_DEG)
fig = plot_surface_xyz_html(cell = cell, elong = ELONG, 
                        polar_deg = POLAR_DEG, azim_deg = AZIM_DEG, roll_deg = ROLL_DEG,
                        out_path = Path("../renders").resolve())